In [2]:
import pandas as pd
from typing import List, Optional
from sqlalchemy import String, Integer, Date, ForeignKey, PrimaryKeyConstraint, text, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy.orm import Session
from typing import Optional
from datetime import date

usuario = "thebridge"
password = "7UBu7Wvi9QX7onkaziGFLPvpuptrzLxe"
host = "dpg-d7agm5idbo4c73a7b800-a.oregon-postgres.render.com" # o la IP del servidor
puerto = "5432"
nombre_db = "thebridge_ivka"

engine = create_engine(f'postgresql://{usuario}:{password}@{host}/{nombre_db}')
# Prueba de conexión
try:
    with engine.connect() as connection:
        print("¡Conexión exitosa a PostgreSQL!")
except Exception as e:
    print(f"Error al conectar: {e}")

¡Conexión exitosa a PostgreSQL!


In [3]:
inspector = inspect(engine)
nombres_tablas = inspector.get_table_names()

for tabla in nombres_tablas:
    print(f"{tabla}")

NameError: name 'inspect' is not defined

In [80]:
class Base(DeclarativeBase):
    pass
class Campus(Base):
    __tablename__ = "campus"

    id_campus: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre_campus: Mapped[str] = mapped_column(String(25), nullable=False)

    # Relaciones
    promociones: Mapped[List["Promocion"]] = relationship(back_populates="campus")


class Vertical(Base):
    __tablename__ = "vertical"

    id_vertical: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre_vertical: Mapped[str] = mapped_column(String(25), nullable=False)

    # Relaciones
    promociones: Mapped[List["Promocion"]] = relationship(back_populates="vertical")
    proyectos: Mapped[List["Proyecto"]] = relationship(back_populates="vertical")


class Proyecto(Base):
    __tablename__ = "proyectos"

    id_proyecto: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre_proyecto: Mapped[str] = mapped_column(String(100), nullable=False)
    id_vertical: Mapped[Optional[int]] = mapped_column(ForeignKey("vertical.id_vertical"))

    # Relaciones
    vertical: Mapped["Vertical"] = relationship(back_populates="proyectos")
    notas: Mapped[List["Nota"]] = relationship(back_populates="proyecto")


class Promocion(Base):
    __tablename__ = "promocion"

    id_promocion: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre_mes: Mapped[str] = mapped_column(String(25), nullable=False)
    fecha_comienzo: Mapped[Optional[date]] = mapped_column(Date)
    id_campus: Mapped[Optional[int]] = mapped_column(ForeignKey("campus.id_campus"))
    id_vertical: Mapped[Optional[int]] = mapped_column(ForeignKey("vertical.id_vertical"))

    # Relaciones
    campus: Mapped["Campus"] = relationship(back_populates="promociones")
    vertical: Mapped["Vertical"] = relationship(back_populates="promociones")
    alumnos: Mapped[List["Alumno"]] = relationship(back_populates="promocion")
    profesores: Mapped[List["Profesor"]] = relationship(back_populates="promocion")


class Alumno(Base):
    __tablename__ = "alumnos"

    id_alumno: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre: Mapped[str] = mapped_column(String(150), nullable=False)
    email: Mapped[str] = mapped_column(String(150), nullable=False)
    id_promocion: Mapped[Optional[int]] = mapped_column(ForeignKey("promocion.id_promocion"))

    # Relaciones
    promocion: Mapped["Promocion"] = relationship(back_populates="alumnos")
    notas: Mapped[List["Nota"]] = relationship(back_populates="alumno")


class Profesor(Base):
    __tablename__ = "profesores"

    id_profesor: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre_profesor: Mapped[str] = mapped_column(String(50), nullable=False)
    rol: Mapped[str] = mapped_column(String(50), nullable=False)
    modalidad: Mapped[str] = mapped_column(String(50), nullable=False)
    id_promocion: Mapped[Optional[int]] = mapped_column(ForeignKey("promocion.id_promocion"))

    # Relaciones
    promocion: Mapped["Promocion"] = relationship(back_populates="profesores")


class Nota(Base):
    __tablename__ = "notas"

    id_alumno: Mapped[int] = mapped_column(ForeignKey("alumnos.id_alumno"), primary_key=True)
    id_proyecto: Mapped[int] = mapped_column(ForeignKey("proyectos.id_proyecto"), primary_key=True)
    nota: Mapped[str] = mapped_column(String(50), nullable=False)

    # Relaciones
    alumno: Mapped["Alumno"] = relationship(back_populates="notas")
    proyecto: Mapped["Proyecto"] = relationship(back_populates="notas")

In [ ]:
nombres_campus = ["Madrid", "Valencia", "Bilbao"]

# 1. Convertimos la lista de strings en una lista de objetos Campus
objetos_campus = [Campus(nombre_campus=nombre) for nombre in nombres_campus]

# 2. Abrimos la sesión y añadimos todo de golpe
with Session(engine) as session:
    session.add_all(objetos_campus)
    session.commit()
    print(f"Se han añadido {len(objetos_campus)} registros.")

Se han añadido 3 registros.


In [104]:
dict_verticales = ["Data Science","Full Stack"]
# 1. Convertimos la lista de strings en una lista de objetos Campus
objetos_verticales = [Vertical(nombre_vertical=nombre) for nombre in dict_verticales]

# 2. Abrimos la sesión y añadimos todo de golpe
with Session(engine) as session:
    session.add_all(objetos_verticales)
    session.commit()
    print(f"Se han añadido {len(objetos_verticales)} registros.")

Se han añadido 2 registros.


In [119]:
with engine.connect() as connection:
    query = text("SELECT * FROM campus LIMIT 5")
    resultado = connection.execute(query)
    
    for fila in resultado:
        print(fila)

(1, 'Madrid')
(2, 'Valencia')
(3, 'Bilbao')


In [ ]:
with engine.connect() as connection:
    query = text("SELECT * FROM campus LIMIT 5")
    resultado = connection.execute(query)
    
    for fila in resultado:
        print(fila)

In [118]:
with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO promocion (nombre_mes, fecha_comienzo, id_campus, id_vertical) VALUES
            ('Septiembre', '2023-09-18', 1, 1),
            ('Febrero', '2024-02-12', 1, 1),
            ('Septiembre', '2023-09-18', 1, 2),
            ('Febrero', '2024-02-12', 2, 2);
        """)
    )

In [12]:
with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO promocion (nombre_mes, fecha_comienzo, id_campus, id_vertical) VALUES
            ('Septiembre', '2024-12-09', 2, 2);
        """)
    )

In [4]:
with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO proyectos (nombre_proyecto, id_vertical) VALUES
            ('Proyecto_HLF', '1'),
            ('Proyecto_EDA', '1'),
            ('Proyecto_BBDD', '1'),
            ('Proyecto_ML', '1'),
            ('Proyecto_Deployment', '1'),
            ('Proyecto_WebDev', '2'),
            ('Proyecto_Frontend', '2'),
            ('Proyecto_Backend', '2'),
            ('Proyecto_React', '2'),
            ('Proyecto_FullStack', '2');
            """)
        )

In [ ]:
with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO proyectos (nombre_proyecto, id_vertical) VALUES
            ('Proyecto_HLF', '1'),
            ('Proyecto_EDA', '1'),
            ('Proyecto_BBDD', '1'),
            ('Proyecto_ML', '1'),
            ('Proyecto_Deployment', '1'),
            ('Proyecto_WebDev', '2'),
            ('Proyecto_Frontend', '2'),
            ('Proyecto_Backend', '2'),
            ('Proyecto_React', '2'),
            ('Proyecto_FullStack', '2');
            """)
        )

In [5]:
with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO alumnos (nombre, email, id_promocion) VALUES
            ('Jafet Casals', 'Jafet_Casals@gmail.com', 1),
            ('Jorge Manzanares', 'Jorge_Manzanares@gmail.com', 1),
            ('Onofre Adadia', 'Onofre_Adadia@gmail.com', 1),
            ('Merche Prada', 'Merche_Prada@gmail.com', 1),
            ('Pilar Abella', 'Pilar_Abella@gmail.com', 1),
            ('Leoncio Tena', 'Leoncio_Tena@gmail.com', 1),
            ('Odalys Torrijos', 'Odalys_Torrijos@gmail.com', 1),
            ('Eduardo Caparrós', 'Eduardo_Caparrós@gmail.com', 1),
            ('Ignacio Goicoechea', 'Ignacio_Goicoechea@gmail.com', 1),
            ('Clementina Santos', 'Clementina_Santos@gmail.com', 1),
            ('Daniela Falcó', 'Daniela_Falcó@gmail.com', 1),
            ('Abraham Vélez', 'Abraham_Vélez@gmail.com', 1),
            ('Maximiliano Menéndez', 'Maximiliano_Menéndez@gmail.com', 1),
            ('Anita Heredia', 'Anita_Heredia@gmail.com', 1),
            ('Eli Casas', 'Eli_Casas@gmail.com', 1),
            ('Guillermo Borrego', 'Guillermo_Borrego@gmail.com', 2),
            ('Sergio Aguirre', 'Sergio_Aguirre@gmail.com', 2),
            ('Carlito Carrión', 'Carlito_Carrión@gmail.com', 2),
            ('Haydée Figueroa', 'Haydée_Figueroa@gmail.com', 2),
            ('Chita Mancebo', 'Chita_Mancebo@gmail.com', 2),
            ('Joaquina Asensio', 'Joaquina_Asensio@gmail.com', 2),
            ('Cristian Sarabia', 'Cristian_Sarabia@gmail.com', 2),
            ('Isabel Ibáñez', 'Isabel_Ibáñez@gmail.com', 2),
            ('Desiderio Jordá', 'Desiderio_Jordá@gmail.com', 2),
            ('Rosalina Llanos', 'Rosalina_Llanos@gmail.com', 2),
            ('Amor Larrañaga', 'Amor_Larrañaga@gmail.com', 3),
            ('Teodoro Alberola', 'Teodoro_Alberola@gmail.com', 3),
            ('Cleto Plana', 'Cleto_Plana@gmail.com', 3),
            ('Aitana Sebastián', 'Aitana_Sebastián@gmail.com', 3),
            ('Dolores Valbuena', 'Dolores_Valbuena@gmail.com', 3),
            ('Julie Ferrer', 'Julie_Ferrer@gmail.com', 3),
            ('Mireia Cabañas', 'Mireia_Cabañas@gmail.com', 3),
            ('Flavia Amador', 'Flavia_Amador@gmail.com', 3),
            ('Albino Macias', 'Albino_Macias@gmail.com', 3),
            ('Ester Sánchez', 'Ester_Sánchez@gmail.com', 3),
            ('Luis Miguel Galvez', 'Luis_Miguel_Galvez@gmail.com', 3),
            ('Loida Arellano', 'Loida_Arellano@gmail.com', 3),
            ('Heraclio Duque', 'Heraclio_Duque@gmail.com', 3),
            ('Herberto Figueras', 'Herberto_Figueras@gmail.com', 3),
            ('Teresa Laguna', 'Teresa_Laguna@gmail.com', 4),
            ('Estrella Murillo', 'Estrella_Murillo@gmail.com', 4),
            ('Ernesto Uriarte', 'Ernesto_Uriarte@gmail.com', 4),
            ('Daniela Guitart', 'Daniela_Guitart@gmail.com', 4),
            ('Timoteo Trillo', 'Timoteo_Trillo@gmail.com', 4),
            ('Ricarda Tovar', 'Ricarda_Tovar@gmail.com', 4),
            ('Alejandra Vilaplana', 'Alejandra_Vilaplana@gmail.com', 4),
            ('Daniel Rosselló', 'Daniel_Rosselló@gmail.com', 4),
            ('Rita Olivares', 'Rita_Olivares@gmail.com', 4),
            ('Cleto Montes', 'Cleto_Montes@gmail.com', 4),
            ('Marino Castilla', 'Marino_Castilla@gmail.com', 4),
            ('Estefanía Valcárcel', 'Estefanía_Valcárcel@gmail.com', 4),
            ('Noemí Vilanova', 'Noemí_Vilanova@gmail.com', 4);
            """)
        )

In [13]:
with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO profesores (nombre_profesor, rol, modalidad, id_promocion) VALUES
            ('Noa Yáñez', 'TA', 'Presencial', 1),
            ('Saturnina Benitez', 'TA', 'Presencial', 1),
            ('Anna Feliu', 'TA', 'Presencial', 3),
            ('Rosalva Ayuso', 'TA', 'Presencial', 5),
            ('Ana Sofia Ferrer', 'TA', 'Presencial', 4),
            ('Angelica Corral', 'TA', 'Presencial', 2),
            ('Ariel Lledo', 'TA', 'Presencial', 1),
            ('Mario Prats', 'LI', 'Online', 4),
            ('Luis Angel Suarez', 'LI', 'Online', 3),
            ('Maria Dolores Diaz', 'LI', 'Online', 1);
            """)
        )

In [ ]:
'''ArithmeticError
with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO Claustro (nombre, rol_id, vertical_id, promocion_id, campus_id, modalidad_id)
            SELECT 
                t.Nombre,
                r.id,
                v.id,
                p.id,
                c.id,
                m.id
            FROM temp_claustro t
            JOIN Roles r ON t.Rol = r.nombre
            JOIN Verticales v ON t.Vertical = v.nombre
            JOIN Promociones p ON t.Promocion = p.nombre
            JOIN Campus c ON t.Campus = c.nombre
            JOIN Modalidades m ON t.Modalidad = m.nombre;
            """)
    ''''